# 06 | LangGraph 基础骨架：State、Node、Edge、Graph 是什么

学习 LangGraph，最先要弄清楚的不是某个高级 API，而是一张图到底由什么组成。

一个最小的 LangGraph 流程，可以先看成这条线：

```text
START -> clean_text -> count_chars -> END
```

这篇只讲基础骨架：`State`、`Node`、`Edge`、`Graph`、`StateGraph`、`START`、`END`。

## 一、从一张最小图开始

下面这个例子不调用模型，也不调用工具，只做两步文本处理：先清洗空格，再统计字符数。

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


# State：定义图中会流动的共享数据结构。
class TextState(TypedDict):
    raw_text: str
    cleaned_text: str
    char_count: int


# Node：读取当前 State，返回要写回 State 的增量更新。
def clean_text(state: TextState) -> dict[str, str]:
    cleaned = state["raw_text"].strip()
    return {"cleaned_text": cleaned}


# Node：继续读取上一步写入的 cleaned_text，再写入 char_count。
def count_chars(state: TextState) -> dict[str, int]:
    return {"char_count": len(state["cleaned_text"])}


# StateGraph：用 TextState 作为状态结构来搭建图。
builder = StateGraph(TextState)

# add_node：把普通 Python 函数注册成图里的节点。
builder.add_node("clean_text", clean_text)
builder.add_node("count_chars", count_chars)

# add_edge：声明节点执行顺序；START 和 END 是图的入口与出口。
builder.add_edge(START, "clean_text")
builder.add_edge("clean_text", "count_chars")
builder.add_edge("count_chars", END)

# compile：把构建器编译成真正可运行的 Graph。
graph = builder.compile()

# invoke：传入初始 State，运行整张图，得到最终 State。
graph.invoke(
    {
        "raw_text": "  LangGraph 让流程显式起来  ",
        "cleaned_text": "",
        "char_count": 0,
    }
)

如果把细节先折叠掉，这段代码其实只做了四件事：

```text
定义 State
 -> 定义 Node
 -> 用 Edge 连接 Node
 -> compile 成 Graph 并运行
```

后面逐个拆开。

## 二、State 与 Node：数据如何被处理

`State` 是整个图共享的数据结构。节点之间不直接把参数传来传去，而是围绕同一份 State 读写数据。

在这个例子里，State 是：

```python
class TextState(TypedDict):
    raw_text: str
    cleaned_text: str
    char_count: int
```

它说明这张图会维护三个字段：

| 字段 | 含义 |
| --- | --- |
| `raw_text` | 原始输入文本 |
| `cleaned_text` | 清洗后的文本 |
| `char_count` | 字符数量 |

LangGraph 官方文档也把 State 放在第一位：它是当前应用快照的共享数据结构，通常用一个状态 schema 描述。这里的 `TypedDict` 就是这个 schema，你可以把它想象成一个数据共享的容器。

### Node：读取 State，返回更新

`Node` 是真正做事的地方。它通常是一个 Python 函数：接收当前 State，返回要写回 State 的字段。

```python
def clean_text(state: TextState) -> dict[str, str]:
    cleaned = state["raw_text"].strip()
    return {"cleaned_text": cleaned}


def count_chars(state: TextState) -> dict[str, int]:
    return {"char_count": len(state["cleaned_text"])}
```

`clean_text` 读取 `raw_text`，返回 `cleaned_text`。

`count_chars` 读取 `cleaned_text`，返回 `char_count`。

注意：节点不需要返回完整 State，只返回这一步要更新的字段即可。LangGraph 会把这些更新合并回 State。

所以这里不写 `-> TextState`。`TextState` 表示完整状态，包含 `raw_text`、`cleaned_text`、`char_count` 三个字段；`clean_text` 只更新 `cleaned_text`，因此返回类型写成 `dict[str, str]` 更贴近实际含义。

## 三、Edge、START 和 END：路线如何连接

`Edge` 负责连接节点，告诉图执行顺序。

这个例子使用的是普通边，也就是固定路线：

```python
builder.add_edge(START, "clean_text")
builder.add_edge("clean_text", "count_chars")
builder.add_edge("count_chars", END)
```

这三行可以直接读成：

```text
从 START 进入 clean_text
clean_text 执行完进入 count_chars
count_chars 执行完到 END 结束
```

`conditional edge` 也是 Edge，但它不是固定路线，而是根据 State 决定下一步走哪条路。这个留到后续篇单独讲。

### START 和 END：入口与出口

`START` 和 `END` 不是业务节点，而是 LangGraph 提供的两个特殊标记。

| 名称 | 作用 |
| --- | --- |
| `START` | 图的入口，表示从哪里开始执行 |
| `END` | 图的出口，表示流程到这里结束 |

没有 `START`，图不知道第一步跑哪个节点；没有 `END`，读代码的人也不容易看出流程在哪里收束。

## 四、StateGraph 和 Graph：从构建到运行

`StateGraph` 可以理解成“图的构建器”。它还不是最终可运行的图，而是用来添加 State、Node 和 Edge。

```python
builder = StateGraph(TextState)
builder.add_node("clean_text", clean_text)
builder.add_node("count_chars", count_chars)
builder.add_edge(START, "clean_text")
builder.add_edge("clean_text", "count_chars")
builder.add_edge("count_chars", END)
```

真正可以运行的是 `compile()` 之后得到的 `graph`：

```python
graph = builder.compile()
```

如果想在 Notebook 里直接看这张图，可以用编译后的 `graph` 对象生成 Mermaid 图片。

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

可以这样区分：

| 名称 | 更像什么 |
| --- | --- |
| `StateGraph` | 施工中的流程图，用来添加节点和边 |
| `Graph` | 编译后的可运行流程，可以 `invoke()` 或 `stream()` |

官方文档也强调：定义完 State、Node、Edge 后，需要先 `compile()`，然后才能使用图。

## 五、把执行过程串起来

现在回到最开始的输入：

```python
inputs = {
    "raw_text": "  LangGraph 让流程显式起来  ",
    "cleaned_text": "",
    "char_count": 0,
}
```

执行链路可以读成：

```text
输入 State
 -> START
 -> clean_text 读取 raw_text，写入 cleaned_text
 -> count_chars 读取 cleaned_text，写入 char_count
 -> END
 -> 返回最终 State
```

这就是 LangGraph 最基础的心智模型：

| 概念 | 一句话理解 |
| --- | --- |
| `State` | 图里流动和积累的数据 |
| `Node` | 读取 State、返回更新的函数 |
| `Edge` | 决定节点之间怎么走 |
| `START` | 图的入口 |
| `END` | 图的出口 |
| `StateGraph` | 组装 State、Node、Edge 的构建器 |
| `Graph` | `compile()` 后得到的可运行流程 |